# 04 · Otimização de hiperparâmetros do Random Forest (Optuna)

**Onde estamos no ML:** entregável 5. Escolhemos o **Random Forest** para otimizar porque ele tem muitos hiperparâmetros e ficou longe do seu limite na comparação (F1-macro 0,852 com valores fixos, abaixo da Regressão Logística). O Optuna procura a configuração que maximiza o F1-macro; tudo está no MLflow (experimento `t1-manter-descartar`).

| Item | Tipo |
| --- | --- |
| Busca bayesiana (TPE) do Optuna | método de otimização, escolhe configurações |
| Random Forest | **ML nosso**, o modelo que decide quais palavras viram sinal |
| Validação cruzada agrupada por frase | regra de avaliação |

## Estratégia

- **Objetivo:** F1-macro **médio** de uma validação cruzada `StratifiedGroupKFold` de 3 dobras, agrupada por frase (palavras da mesma frase nunca ficam em dobras diferentes).
- **Por que uma amostra:** a busca roda sobre uma amostra de **6.000 frases inteiras** (~76 mil palavras), porque cada tentativa treina 3 florestas; usar as 1,3 milhão de palavras tornaria inviável. O modelo final é refeito nos dados completos.
- **Método:** TPE (otimização bayesiana) com semente fixa, máximo de 25 tentativas e 40 minutos.
- **Sem vazamento:** o teste final (10% das frases) não participou; a comparação final usa a mesma validação dos outros modelos.

### Espaço de busca

| Hiperparâmetro | Intervalo |
| --- | --- |
| `n_estimators` | inteiro 50–200 |
| `max_depth` | inteiro 10–60 |
| `min_samples_leaf` | inteiro 1–30 (escala log) |
| `max_features` | `sqrt` ou fração 0,02–0,5 (escala log) |
| `class_weight` | `balanced`, `balanced_subsample` ou nenhum |
| `criterion` | `gini` ou `entropy` |

In [1]:
import os, sys, json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import mlflow
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from textgloss import train, tune
URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(URI); mlflow.set_experiment(train.EXPERIMENT)
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name(train.EXPERIMENT).experiment_id
estudo = client.search_runs([experiment], filter_string="tags.modelo = 'optuna-estudo' and attributes.status = 'FINISHED'",
                            order_by=["attributes.start_time DESC"])[0]
filhos = client.search_runs([experiment], filter_string=f"tags.mlflow.parentRunId = '{estudo.info.run_id}'", max_results=500)
runs = train.latest_runs()
def mostrar(run_id, caminho):
    arquivo = mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{run_id}/{caminho}", tracking_uri=URI)
    plt.figure(figsize=(8, 4.5)); plt.imshow(mpimg.imread(arquivo)); plt.axis("off"); plt.tight_layout()
print(f"estudo {estudo.info.run_id[:8]} · {len(filhos)} trials · MLflow: {URI}")

estudo d7fd13cc · 25 trials · MLflow: http://localhost:5000


## 1. Todas as tentativas

In [2]:
linhas = []
for r in filhos:
    p, m = r.data.params, r.data.metrics
    linhas.append({"trial": r.data.tags["mlflow.runName"], **{k: p.get(k) for k in ["n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight", "criterion"]},
                   "f1_macro_cv": m.get("f1_macro_cv"), "desvio_dobras": m.get("f1_macro_cv_desvio"), "tempo_s": m.get("tempo_s")})
trials = pd.DataFrame(linhas).sort_values("trial").reset_index(drop=True)
print(f"{len(trials)} tentativas · F1-macro CV: mínimo {trials.f1_macro_cv.min():.4f}, mediana {trials.f1_macro_cv.median():.4f}, máximo {trials.f1_macro_cv.max():.4f}")
trials.sort_values("f1_macro_cv", ascending=False).round(4).head(10)

25 tentativas · F1-macro CV: mínimo 0.8377, mediana 0.8645, máximo 0.8712


,trial,n_estimators,max_depth,min_samples_leaf,max_features,class_weight,criterion,f1_macro_cv,desvio_dobras,tempo_s
13,trial-013,154,56,3,0.15121522455518785,balanced,gini,0.8712,0.0021,19.0350
14,trial-014,187,44,4,0.14502798374899525,balanced,gini,0.8698,0.0026,20.0954
23,trial-023,79,52,3,0.09724675523330942,balanced,gini,0.8696,0.0034,7.5557
10,trial-010,98,57,5,0.3710290707365666,balanced,gini,0.8695,0.0026,18.9406
11,trial-011,121,55,7,0.34107772059939206,balanced,gini,0.8693,0.0026,22.5355
20,trial-020,196,43,2,0.21831858072571755,balanced,gini,0.8693,0.0023,28.5553
16,trial-016,181,45,4,0.11702415807857079,balanced,entropy,0.8691,0.0019,18.1424
0,trial-000,58,54,6,0.21101513838453084,balanced,gini,0.8690,0.0024,8.6621
15,trial-015,195,34,5,0.13689375572680026,balanced,gini,0.8683,0.0031,17.8549
22,trial-022,190,53,9,0.24109578545044047,balanced,gini,0.8673,0.0023,25.2451


## 2. Histórico da otimização

In [3]:
mostrar(estudo.info.run_id, "historico_otimizacao.png")

Cada ponto é uma tentativa e a linha mostra o melhor valor até então. Se a curva sobe e estabiliza, a busca encontrou uma região boa; se ainda sobe no fim, mais tentativas ajudariam.

## 3. Quais hiperparâmetros importam

In [4]:
mostrar(estudo.info.run_id, "importancia_hiperparametros.png")

## 4. Melhores parâmetros e resultado final

In [5]:
melhores = json.load(open(mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{estudo.info.run_id}/melhores_parametros.json", tracking_uri=URI)))
print(json.dumps(melhores, indent=2)); print("melhor F1-macro na CV (amostra):", round(estudo.data.metrics["melhor_f1_macro_cv"], 4))

{
  "n_estimators": 154,
  "max_depth": 56,
  "min_samples_leaf": 3,
  "max_features": 0.15121522455518785,
  "class_weight": "balanced",
  "criterion": "gini"
}
melhor F1-macro na CV (amostra): 0.8712


In [6]:
ordem = ["random-forest", "random-forest-otimizado", "arvore-de-decisao", "regressao-logistica"]
linhas = []
for n in ordem:
    m = runs[n].data.metrics
    linhas.append({"modelo": n, "f1_macro_val": m["val_f1_macro"], "roc_auc": m["val_roc_auc"], "pr_auc": m["val_pr_auc"], "rec_descartar": m["val_recall_descartar"],
                   "f1_treino": m["treino_f1_macro"], "gap_treino_val": m["gap_f1_macro_treino_val"], "treino_s": m["tempo_treino_s"], "predicao_s": m["tempo_predicao_val_s"]})
comp = pd.DataFrame(linhas).set_index("modelo"); comp.round(4)

,f1_macro_val,roc_auc,pr_auc,rec_descartar,f1_treino,gap_treino_val,treino_s,predicao_s
modelo,,,,,,,,
random-forest,0.8524,0.9207,0.9112,0.7698,0.8532,0.0008,98.3464,0.8129
random-forest-otimizado,0.8840,0.9466,0.9394,0.8325,0.8996,0.0156,1520.3648,1.0810
arvore-de-decisao,0.8703,0.9139,0.8835,0.8128,0.8795,0.0092,86.3755,0.4403
regressao-logistica,0.8759,0.9442,0.9422,0.8243,0.8770,0.0011,40.8371,0.4368


In [7]:
fig, ax = plt.subplots(figsize=(7, 3)); comp["f1_macro_val"].sort_values().plot.barh(ax=ax, color="#4c78a8"); ax.set_xlim(0.84, 0.89); ax.set_xlabel("F1-macro (validação)")
for i, v in enumerate(comp["f1_macro_val"].sort_values()): ax.text(v + 0.0008, i, f"{v:.4f}", va="center")
plt.tight_layout()

## 5. Importância dos atributos do modelo otimizado

In [8]:
mostrar(runs["random-forest-otimizado"].info.run_id, "importancia_atributos.png")

## 6. Análise

**1. A otimização valeu a pena.** O Random Forest foi de **0,852 para 0,884** de F1-macro na validação (+3,2 pontos) e passou a Regressão Logística (0,876) e a Árvore (0,870). ROC-AUC foi de 0,921 para 0,947. A revocação de `DESCARTAR`, que era a fraqueza do modelo, subiu de 0,770 para 0,832.

**2. O que mudou foi principalmente `max_features`.** O padrão `sqrt` (cerca de 2% das ~2.000 colunas por divisão) deixa cada divisão quase sem atributos úteis num espaço one-hot esparso. As tentativas com `max_features` entre 10% e 40% das colunas dominam o topo, e as de `sqrt` ficam entre 0,838 e 0,852. Isso confirma a hipótese levantada na comparação: o Random Forest estava mal configurado, não era inferior. Árvores profundas (44–57) com folhas pequenas (2–7 amostras) e `class_weight="balanced"` completam o perfil dos melhores.

**3. As melhores tentativas são praticamente empatadas.** As dez melhores ficam entre 0,867 e 0,871 de F1-macro, com desvio de 0,002 a 0,003 entre dobras. Diferenças nessa ordem estão dentro do ruído, então a configuração escolhida (tentativa 13) é *uma* boa região, não "a" configuração ótima. O histórico mostra que a busca estabilizou rápido; mais tentativas trariam ganho pequeno.

**4. O custo é real.** O treino final levou cerca de 25 minutos (contra 98 s do padrão e 41 s da regressão), por causa de 154 árvores profundas com muitas colunas por divisão. A predição continua barata (1,1 s para 146 mil palavras), o que importa para a API. O modelo também é bem maior em memória.

**5. Sinal moderado de sobreajuste.** F1-macro de 0,900 no treino contra 0,884 na validação (diferença de 0,016), contra 0,001 da regressão. Não é grave, mas é a primeira vez que um modelo nosso mostra a diferença; a fase 6 vai olhar isso com validação cruzada e curva de aprendizado.

**6. Cuidados com a conclusão.**
- A vantagem sobre a Regressão Logística é de **0,8 ponto numa única validação**. Só a validação cruzada da fase 6 diz se é estável.
- A busca usou uma amostra de 76 mil palavras (F1 de CV 0,871), enquanto o modelo final usa 1,3 milhão (0,884): mais dados ajudam, e os parâmetros ótimos para a amostra podem não ser os ótimos para o conjunto completo.
- Os rótulos ainda têm ruído não medido (conferência manual pendente), o que limita o teto de todos os modelos.
- O teste final continua intocado.

**7. Implicação para a escolha do modelo.** O Random Forest otimizado é o melhor em F1-macro, mas a Regressão Logística é mais de 35 vezes mais rápida de treinar, tem ROC-AUC e PR-AUC equivalentes e é explicável. A escolha do modelo que vai para a API deve esperar a validação cruzada e pesar custo, estabilidade e explicabilidade, não só o F1 de uma validação.
